In [ ]:
from dotenv import load_dotenv
from langchain_classic.document_loaders import TextLoader
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_classic.storage import LocalFileStore
from langchain_classic.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

"""
[Assignment 04] RAG with Stuff Documents Chain (26-05-17)

- Stuff Documents 체인을 사용하여 완전한 RAG 파이프라인을 구현하세요.
- 체인을 수동으로 구현해야 합니다.
- 체인에 ConversationBufferMemory를 부여합니다.
- 이 문서를 사용하여 RAG를 수행하세요: https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223
- 체인에 다음 질문을 합니다:
  - Aaronson 은 유죄인가요?
  - 그가 테이블에 어떤 메시지를 썼나요?
  - Julia 는 누구인가요?
"""

load_dotenv()

# 0. LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 1. 문서 로드하기 (load)
loader = TextLoader("./files/document.txt", encoding="utf-8")
documents = loader.load()


# 2. 문서 쪼개기 (transform)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#   separator="\n",
  chunk_size=1200, # 1200자 청크로 분할
  chunk_overlap=300, # 청크 간 300자 겹침으로 문맥 유지
)

splitted_docs = splitter.split_documents(documents)


# 3. 임베딩 생성 및 캐시
cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)


# 4. 벡터 스토어 생성
vectorstore = FAISS.from_documents(splitted_docs, cached_embeddings)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 12}, # 벡터 유사도 검색 시 가장 유사한 12개 문서 반환
) 


# 5. 대화 메모리와 질문 처리
MEMORY_KEY = "chat_history"
INPUT_KEY = "question"
OUTPUT_KEY = "answer"

memory = ConversationBufferMemory(
    memory_key=MEMORY_KEY,
    input_key=INPUT_KEY,
    output_key=OUTPUT_KEY,
    return_messages=True,
)


# 6. 체인 연결 (StuffDocumentsChain 직접 구현)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def load_memory(_):
    return memory.load_memory_variables({})[MEMORY_KEY]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an AI assistant performing document-based question answering.
        You must answer only using the provided Context.
        Do not speculate on information not present in the Context.
        If you do not know the answer, respond only with "I don't know."

        Rules:
        - Answer in English.
        - Keep answers short and clear.
        - Refer to the chat history if it is relevant.

        Context:
        {context}
        """
    ),
    MessagesPlaceholder(variable_name=MEMORY_KEY),
    ("human", "{question}")
])

chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        MEMORY_KEY: RunnableLambda(load_memory),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

# 7. 메모리 저장까지 포함한 수동 체인
def ask(question: str):
    response = chain.invoke(question)
    answer = response.content
    memory.save_context(
        {INPUT_KEY: question},
        {OUTPUT_KEY: answer},
    )

    return answer

In [8]:
# 8. 질문 실행
questions = [
    "Is Aaronson guilty?",
    # "What did Winston write in the dust on the table?",
    "What message did he write in the table?",
    "Who is Julia?"
]

for q in questions:
    print("Q:", q)
    print("A:", ask(q))
    print()

Q: Is Aaronson guilty?
A: Winston believes that Jones, Aaronson, and Rutherford are guilty of the crimes they were charged with, and he has convinced himself that he had never seen the photograph that disproved their guilt.

Q: What message did he write in the table?
A: Winston wrote "2+2=5" in the dust on the table.

Q: Who is Julia?
A: Julia is a character with whom Winston has a romantic relationship. She is someone he loves and has shared intimate moments with, but their relationship is complicated by the oppressive regime they live under.

